# 5. Longest Palindromic Substring
**Difficulty:** 🟡 Medium · **Topic:** String · **LeetCode:** https://leetcode.com/problems/longest-palindromic-substring/

## 💡 Concepts

**Core concept(s):** **Expand around center**; also solvable with **dynamic programming**.

**Why it applies here:** Every palindrome grows out from a middle. Try each possible middle (each character, and each gap between characters) and stretch outward while both sides match — the longest stretch wins.

**Key intuition:** A palindrome is symmetric around its center, so plant a center and grow outward while the mirror holds.

---

### 📚 What is "Expand Around Center"?
Every palindrome has a middle. **Expand around center** places a middle (a single character, or the gap between two) and walks outward while the two sides match.
- **Complexity:** each of the ~2n centers can expand up to n steps → **O(n²)** time, **O(1)** space.

### 📚 What is Dynamic Programming (DP)?
**DP** reuses answers to smaller subproblems instead of recomputing them. Here: "is `s[i..j]` a palindrome?" is true when the ends match **and** the inside `s[i+1..j-1]` is already known to be a palindrome.
- **Complexity:** fill an n×n table once → **O(n²)** time and space.

---

**Prerequisite knowledge:**
- Expanding from a center with two pointers.
- A 2-D DP table of "is s[i..j] a palindrome?".

## 📝 Problem

Return the longest substring of `s` that is a palindrome.

**Example**
```
"babad" -> "bab"  (or "aba")
"cbbd"  -> "bb"
```

> Three approaches: brute `O(n^3)`, DP `O(n^2)`, and expand-around-center `O(n^2)` with `O(1)` space.

### Approach 1 — Check Every Substring (worst)

**Idea:** Try all substrings; keep the longest that reads the same reversed.

**Time complexity:** `O(n^3)` — `O(n^2)` substrings, each `O(n)` to check.

**Space complexity:** `O(1)` extra.

In [ ]:
def lps_brute(s: str) -> str:
    best = ""
    n = len(s)
    for i in range(n):                     # start of the substring
        for j in range(i, n):              # end of the substring
            # Only bother checking if this substring is longer than our current best.
            if j - i + 1 > len(best) and s[i:j+1] == s[i:j+1][::-1]:
                best = s[i:j+1]            # it's a longer palindrome -> keep it
    return best

### Approach 2 — Dynamic Programming (better)

**Idea:** `dp[i][j]` = is `s[i..j]` a palindrome? True when the ends match and the inside `dp[i+1][j-1]` is a palindrome. Build from short spans to long.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(n^2)` for the table.

In [ ]:
def lps_dp(s: str) -> str:
    n = len(s)
    if n < 2:
        return s                           # empty or single char is already a palindrome
    dp = [[False] * n for _ in range(n)]   # dp[i][j] = is s[i..j] a palindrome?
    start, maxlen = 0, 1                   # best palindrome found so far (position + length)
    for i in range(n):
        dp[i][i] = True                    # every single character is a palindrome
    for length in range(2, n + 1):         # build up from length 2 to n
        for i in range(n - length + 1):    # left index of the window
            j = i + length - 1             # right index of the window
            # It's a palindrome if the ends match AND the inside is already a palindrome.
            if s[i] == s[j] and (length == 2 or dp[i+1][j-1]):
                dp[i][j] = True
                if length > maxlen:        # remember the longest one seen
                    start, maxlen = i, length
    return s[start:start + maxlen]

### Approach 3 — Expand Around Center (optimal space)

**Idea:** For each center (single char for odd lengths, gap for even), expand outward while sides match. Track the longest.

**Time complexity:** `O(n^2)`.

**Space complexity:** `O(1)`.

In [ ]:
def longest_palindrome(s: str) -> str:
    if not s:
        return ""

    best = ""

    def expand(left, right):
        # Keep expanding while inside string and both sides equal
        while left >= 0 and right < len(s) and s[left] == s[right]:
            left -= 1
            right += 1
        # After loop, left/right went one step too far — fix and return substring
        return s[left + 1 : right]

    for i in range(len(s)):
        # Odd: center at i   e.g. "aba"
        odd = expand(i, i)

        # Even: center between i and i+1   e.g. "abba"
        even = expand(i, i + 1)

        # Keep whichever is longer
        if len(odd) > len(best):
            best = odd
        if len(even) > len(best):
            best = even

    return best

In [ ]:
# def lps_expand(s: str) -> str:
#     if not s:
#         return ""
#     start, end = 0, 0                      # bounds of the best palindrome found
#     def grow(l, r):                        # expand outward from a center while sides match
#         while l >= 0 and r < len(s) and s[l] == s[r]:
#             l -= 1; r += 1                 # step both ends outward
#         return l + 1, r - 1                # step back to the last valid (matching) bounds
#     for i in range(len(s)):
#         l1, r1 = grow(i, i)                # odd-length palindrome centered on i
#         l2, r2 = grow(i, i + 1)            # even-length palindrome centered between i and i+1
#         if r1 - l1 > end - start:          # found a longer odd palindrome?
#             start, end = l1, r1
#         if r2 - l2 > end - start:          # found a longer even palindrome?
#             start, end = l2, r2
#     return s[start:end + 1]

In [ ]:
# Correctness check (answers may differ but must be palindromes of the right length)
def ok(s, out):
    return out == out[::-1] and out in s
tests = [("babad",3), ("cbbd",2), ("a",1), ("ac",1), ("forgeeksskeegfor",10)]
for s, ln in tests:
    a, b, c = lps_brute(s), lps_dp(s), lps_expand(s)
    print(f"{s!r:>18} -> brute={a!r}, dp={b!r}, expand={c!r}")
    assert len(a) == len(b) == len(c) == ln and ok(s,a) and ok(s,b) and ok(s,c), "mismatch!"
print("\nAll tests passed")

## ⏱️ Empirically Checking the Complexities

Big-O can't be read off a function directly, but it can be **measured**. We time each approach on inputs of growing `n` and read the **doubling ratio** — how much runtime grows when `n` doubles.

| Theoretical | Ratio when `n` → `2n` |
|-------------|-----------------------|
| `O(n)`        | ≈ **2×** |
| `O(n log n)`  | ≈ **2×** (slightly more) |
| `O(n²)`       | ≈ **4×** |
| `O(n³)`       | ≈ **8×** |

Inputs are built to force the **worst case** (no early exit). Sub-millisecond rows are noisy — look at the trend.

In [ ]:
import os, sys
_root = os.getcwd()
for _ in range(5):
    if os.path.exists(os.path.join(_root, "bench_utils.py")):
        break
    _root = os.path.dirname(_root)
if _root not in sys.path:
    sys.path.insert(0, _root)
from bench_utils import benchmark   # shared: prints ratio table + optional log-log plot

def make_worst_case(n):
    s = "a" * n                             # every substring is a palindrome -> worst case
    return (s,)

solutions = {
    "brute  O(n^3)": lps_brute,
    "dp     O(n^2)": lps_dp,
    "expand O(n^2)": lps_expand,
}
sizes = [40, 80, 160, 320]                  # small: brute is cubic

benchmark(solutions, make_worst_case, sizes, plot=True)


## 🧩 Patterns Learned

- **Expand around center:** for palindrome problems, grow outward from each of the ~2n centers — `O(n^2)` time, `O(1)` space, and easy to write.
- **DP on substrings:** "is this span valid?" often reduces to "ends match AND the inside is valid".
- **Signal:** "longest/any palindrome", "symmetric substring".
- **Related problems:** Palindromic Substrings, Longest Palindromic Subsequence, Valid Palindrome.
- **Common pitfalls:** (1) handling only odd centers (forgetting even-length palindromes); (2) off-by-one when returning the final bounds.